# Day 2 | Notebook 4: Production RAG Pipeline
**RedisVL Focus:** `HFTextVectorizer`, `SemanticCache`, `SearchIndex` (PDR), `SemanticRouter`

---
## 📌 Learning Objectives
1. Understand the Parent-Document Retrieval (PDR) pattern
2. Build a Semantic Cache to skip LLM calls for repeated questions
3. Implement Intent Routing before retrieval
4. Wire all three into a complete, layered production pipeline
---

In [1]:
# ─── INSTRUCTOR SETTINGS ─────────────────────────────────────────
SHOW_INSIGHTS = False
REDIS_URL = "redis://redis-vector-db:6379"
# ─────────────────────────────────────────────────────────────────
import time
from redisvl.index import SearchIndex
from redisvl.schema import IndexSchema
from redisvl.query import VectorQuery
from redisvl.extensions.llmcache import SemanticCache
from redisvl.utils.vectorize import HFTextVectorizer

vectorizer = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ HFTextVectorizer loaded (all-MiniLM-L6-v2, 384 dims)")

/tmp/ipykernel_79405/4177169385.py:9: DeprecationWarning: Importing from redisvl.extensions.llmcache is deprecated. Please import from redisvl.extensions.cache.llm instead.
  from redisvl.extensions.llmcache import SemanticCache


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ HFTextVectorizer loaded (all-MiniLM-L6-v2, 384 dims)


## 📌 Concept: Parent-Document Retrieval (PDR)

```
PROBLEM with naive chunking:
  Chunk: 'cooled GPU. The battery' ← crosses sentence boundary, loses meaning

PDR SOLUTION — Two levels:
  Child (indexed): atomic proposition  → small, precise, searchable
  Parent (stored): full paragraph      → rich context for the LLM

  Example:
  Input: 'The Aether X1 has RTX 4090 with liquid cooling. Price is $2,499.'

  Child propositions (indexed in Redis):
    → 'Aether X1 has RTX 4090 with liquid cooling'  (parent_id='doc_aether_x1')
    → 'Aether X1 price is $2,499'                   (parent_id='doc_aether_x1')

  Query: 'What GPU does the Aether X1 have?'
  → Matches child: 'Aether X1 has RTX 4090...'
  → Returns parent: full paragraph with all details
```


In [2]:
# Cell 1: PDR index schema
PDR_SCHEMA = {
    "index": {"name": "nb04_pdr", "prefix": "chunk:", "storage_type": "json"},
    "fields": [
        {"name": "parent_id",   "type": "tag"},
        {"name": "proposition", "type": "text"},
        {"name": "embedding",   "type": "vector",
         "attrs": {"dims": 384, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
}
pdr_index = SearchIndex(IndexSchema.from_dict(PDR_SCHEMA), redis_url=REDIS_URL)
pdr_index.create(overwrite=True)
print("✅ PDR index created")

✅ PDR index created


In [3]:
# Cell 2: Knowledge base — 6 product documents
KNOWLEDGE_BASE = {
    "doc_aether_x1": (
        "The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary "
        "liquid-cooling system. It has a 4K OLED 120Hz ProMotion display, 64GB DDR5 RAM, "
        "and 2TB NVMe SSD. The price is $2,499 with a 3-year global warranty."
    ),
    "doc_sony_xm5": (
        "The Sony WH-1000XM5 headphones feature industry-leading active noise cancellation "
        "with dual-chip processing. Battery life is 30 hours with ANC on, 40 hours without. "
        "Supports LDAC, AAC, and SBC codecs. Price is $349."
    ),
    "doc_nebula_tab": (
        "The Nebula Tab Pro is a 12-inch OLED tablet with an Apple M3 chip. "
        "It supports the Aether Pencil 2nd generation and the Magic Keyboard. "
        "Storage options: 256GB, 512GB, 1TB. Starting price $899."
    ),
    "doc_warranty": (
        "All Aether products come with a 3-year global warranty covering manufacturing defects. "
        "Accidental damage is covered under AetherCare+ for an additional $99/year. "
        "Warranty claims can be filed at any Aether Store or via the online portal."
    ),
    "doc_shipping": (
        "Standard shipping takes 3-5 business days. Express delivery (1-2 days) is available "
        "for $15. Free shipping applies to orders over $500. "
        "International shipping is available to 45 countries."
    ),
    "doc_returns": (
        "Products can be returned within 30 days of purchase in original condition. "
        "Digital downloads and opened software are non-refundable. "
        "Refunds are processed within 5-7 business days after return receipt."
    ),
}

print(f"✅ Knowledge base: {len(KNOWLEDGE_BASE)} documents")

✅ Knowledge base: 6 documents


In [4]:
# Cell 3: Propositional decomposition — split text into atomic facts
def extract_propositions(text: str) -> list[str]:
    """Split text into atomic propositions (sentence-level facts)."""
    raw = [s.strip() for s in text.replace('\n', ' ').split('. ')]
    return [s + '.' if not s.endswith('.') else s for s in raw if len(s) > 15]

# Demo
doc = KNOWLEDGE_BASE["doc_aether_x1"]
props = extract_propositions(doc)
print("📌 Propositional Decomposition — doc_aether_x1:")
for i, p in enumerate(props, 1):
    print(f"  [{i}] {p}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Each proposition is now independently searchable.")
    print("   'RTX 4090 GPU' maps to prop [1]. '$2,499 price' maps to prop [3].")
    print("   Both share parent_id='doc_aether_x1' so we return the full paragraph.")

📌 Propositional Decomposition — doc_aether_x1:
  [1] The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling system.
  [2] It has a 4K OLED 120Hz ProMotion display, 64GB DDR5 RAM, and 2TB NVMe SSD.
  [3] The price is $2,499 with a 3-year global warranty.


In [5]:
# Cell 4: Ingest all documents into PDR index
all_docs = []
parent_store = {}   # In-memory parent store: parent_id → full_text

for parent_id, full_text in KNOWLEDGE_BASE.items():
    parent_store[parent_id] = full_text
    propositions = extract_propositions(full_text)
    embeddings = vectorizer.embed_many(propositions, as_buffer=False)
    for prop, emb in zip(propositions, embeddings):
        all_docs.append({"parent_id": parent_id, "proposition": prop, "embedding": emb})

pdr_index.load(all_docs)
print(f"✅ Indexed {len(all_docs)} propositions from {len(KNOWLEDGE_BASE)} documents")
print(f"   Average propositions per doc: {len(all_docs)/len(KNOWLEDGE_BASE):.1f}")

✅ Indexed 20 propositions from 6 documents
   Average propositions per doc: 3.3


In [22]:
# Cell 5: PDR query — match a proposition, return the full parent
def pdr_query(question: str, threshold: float = 0.35) -> dict:
    q_vec = vectorizer.embed(question, as_buffer=False)
    vq = VectorQuery(
        vector=q_vec,
        vector_field_name="embedding",
        return_fields=["parent_id", "proposition"],
        num_results=1
    )
    results = pdr_index.query(vq)
    if not results:
        return {"found": False}
    r = results[0]
    dist = float(r["vector_distance"])
    if dist > threshold:
        return {"found": False, "distance": dist}
    return {
        "found": True,
        "matched_proposition": r["proposition"],
        "parent_id": r["parent_id"],
        "full_context": parent_store[r["parent_id"]],
        "confidence": round(1.0 - dist, 4),
        "distance": dist
    }

result = pdr_query("What GPU does the Aether X1 have?")
print("🔍 PDR Query: 'What GPU does the Aether X1 have?'")
print(f"   Matched proposition : {result['matched_proposition']}")
print(f"   Distance            : {result['distance']:.4f}")
print(f"   Confidence          : {result['confidence']}")
print(f"   Full context (100c) : {result['full_context'][:100]}...")

🔍 PDR Query: 'What GPU does the Aether X1 have?'
   Matched proposition : The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling system.
   Distance            : 0.2170
   Confidence          : 0.783
   Full context (100c) : The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling syst...


In [66]:
pdr_query("What is the Sony battery life?", threshold=0.48)

{'found': True,
 'matched_proposition': 'Battery life is 30 hours with ANC on, 40 hours without.',
 'parent_id': 'doc_sony_xm5',
 'full_context': 'The Sony WH-1000XM5 headphones feature industry-leading active noise cancellation with dual-chip processing. Battery life is 30 hours with ANC on, 40 hours without. Supports LDAC, AAC, and SBC codecs. Price is $349.',
 'confidence': 0.5362,
 'distance': 0.463777899742}

## 📌 Concept: Semantic Cache

Running an LLM for every question is expensive and slow.
Many user questions are **semantically similar** (same intent, different words):
```
'What does the Aether X1 cost?'
'How much is the Aether X1?'     ← same question!
'Price of Aether X1?'            ← same question!
```

**Semantic Cache** stores (question_vector → answer) pairs.
On new questions, it checks: is this vector close to a cached question?
If yes → return cached answer instantly (< 10ms, no LLM call).


In [53]:
# Cell 6: Create Semantic Cache
cache = SemanticCache(
    name="nb04_cache",
    redis_url=REDIS_URL,
    distance_threshold=0.35,   # lower = stricter match required
    vectorizer=vectorizer,
    ttl=300,
    overwrite=True
)
print("✅ Semantic cache created")
print(f"   Threshold: {cache.distance_threshold} (questions within this distance = same intent)")
print(f"   TTL: 300 seconds")

✅ Semantic cache created
   Threshold: 0.35 (questions within this distance = same intent)
   TTL: 300 seconds


In [54]:
# Cell 7: Simulate slow LLM + cache store
def simulate_llm(question: str, context: str) -> str:
    time.sleep(0.5)   # simulate 500ms LLM call
    return f"Based on our documentation: {context[:200].strip()}..."

QUESTION_1 = "How much does the Aether X1 cost?"

# Check cache first
t0 = time.perf_counter()
cached = cache.check(prompt=QUESTION_1)
if cached:
    print("CACHE HIT:", cached[0]['response'][:80])
else:
    # Cache miss → PDR + LLM
    ctx = pdr_query(QUESTION_1, threshold=0.45)
    answer = simulate_llm(QUESTION_1, ctx['full_context']) if ctx['found'] else "Not found."
    cache.store(prompt=QUESTION_1, response=answer, metadata={"parent_id": ctx.get('parent_id','')})
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"CACHE MISS → LLM called ({elapsed:.0f}ms)")
    print(f"  Answer: {answer[:100]}...")

CACHE MISS → LLM called (593ms)
  Answer: Based on our documentation: The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a pro...


In [55]:
## cache delete index
# cache.delete()

In [56]:
# Cell 8: Prove the cache works with rephrased questions
REPHRASINGS = [
    "What is the price of the Aether X1?",
    "Aether X1 price?",
    "How expensive is the Aether X1 laptop?",
    "Tell me the cost of the Aether X1."
]

print(f"{'Question':45} {'Result':10} {'Time (ms)':>10}")
print("-" * 70)
for q in REPHRASINGS:
    t0 = time.perf_counter()
    hit = cache.check(prompt=q)
    elapsed = (time.perf_counter() - t0) * 1000
    status = "HIT ✅" if hit else "MISS ❌"
    print(f"{q:45} {status:10} {elapsed:>10.1f}ms")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: 'Aether X1 price?' has very different tokens from the original")
    print("   but the VECTOR is close — same concept. That's semantic caching.")

Question                                      Result      Time (ms)
----------------------------------------------------------------------
What is the price of the Aether X1?           HIT ✅            56.7ms
Aether X1 price?                              HIT ✅            36.6ms
How expensive is the Aether X1 laptop?        HIT ✅            12.9ms
Tell me the cost of the Aether X1.            HIT ✅             7.0ms


## 📌 Concept: Intent Routing

Before searching, classify the question into a domain:
```
'My GPU is overheating'      → technical_support  (→ search support docs)
'When does my order arrive?' → shipping           (→ search shipping docs)
'Can I return this?'         → returns            (→ search return policy)
```
This prevents cross-domain hallucination: a warranty question
should NOT search the shipping index.


In [57]:
# Cell 9: Simple intent router using vector similarity
INTENT_EXAMPLES = {
    "product_specs": [
        "What are the specs?", "Does it have a GPU?", "How much RAM?"
    ],
    "pricing": [
        "How much does it cost?", "What is the price?", "Is there a discount?"
    ],
    "technical_support": [
        "My device is overheating", "Screen is flickering", "It won't turn on"
    ],
    "shipping": [
        "When will it arrive?", "Track my order", "Express delivery available?"
    ],
    "returns": [
        "Can I return this?", "Refund policy?", "30 day return?"
    ],
    "warranty": [
        "What does warranty cover?", "How long is the warranty?", "AetherCare+?"
    ]
}

# Build intent index: embed one representative example per intent
INTENT_SCHEMA = {
    "index": {"name": "nb04_intents", "prefix": "intent:", "storage_type": "json"},
    "fields": [
        {"name": "intent",    "type": "tag"},
        {"name": "embedding", "type": "vector",
         "attrs": {"dims": 384, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
}
intent_index = SearchIndex(IndexSchema.from_dict(INTENT_SCHEMA), redis_url=REDIS_URL)
intent_index.create(overwrite=True)

intent_docs = []
for intent, examples in INTENT_EXAMPLES.items():
    for ex in examples:
        intent_docs.append({
            "intent": intent,
            "embedding": vectorizer.embed(ex, as_buffer=False)
        })
intent_index.load(intent_docs)
print(f"✅ Intent router seeded ({len(intent_docs)} examples across {len(INTENT_EXAMPLES)} intents)")

✅ Intent router seeded (18 examples across 6 intents)


In [58]:
# Cell 10: Route queries to the correct intent
def detect_intent(question: str) -> str:
    q_vec = vectorizer.embed(question, as_buffer=False)
    vq = VectorQuery(q_vec, "embedding", return_fields=["intent"], num_results=1)
    results = intent_index.query(vq)
    return results[0]["intent"] if results else "general"

TEST_QUESTIONS = [
    "What is the battery life of the Sony XM5?",
    "My Aether X1 keeps shutting down unexpectedly.",
    "How long does standard delivery take?",
    "Is the RTX 4090 really liquid cooled?",
    "Does the 3-year warranty cover accidental drops?",
    "I want to return my purchase",
]

print(f"{'Question':50} {'Intent':20}")
print("-" * 72)
for q in TEST_QUESTIONS:
    intent = detect_intent(q)
    print(f"{q:50} {intent:20}")

Question                                           Intent              
------------------------------------------------------------------------
What is the battery life of the Sony XM5?          warranty            
My Aether X1 keeps shutting down unexpectedly.     warranty            
How long does standard delivery take?              shipping            
Is the RTX 4090 really liquid cooled?              product_specs       
Does the 3-year warranty cover accidental drops?   warranty            
I want to return my purchase                       returns             


In [67]:
# Cell 11: Full 3-layer production pipeline
GUARD_THRESHOLD = 0.48

def production_rag(question: str) -> dict:
    """
    Layer 1: Intent routing
    Layer 2: Semantic cache check
    Layer 3: PDR retrieval → LLM simulation → faithfulness guard → cache store
    """
    t0 = time.perf_counter()

    # Layer 1: Intent
    intent = detect_intent(question)

    # Layer 2: Cache check
    cached = cache.check(prompt=question)
    if cached:
        return {"answer": cached[0]["response"], "source": "cache",
                "intent": intent, "cache_hit": True,
                "latency_ms": round((time.perf_counter() - t0) * 1000, 1)}

    # Layer 3: PDR retrieval
    pdr_result = pdr_query(question, threshold=GUARD_THRESHOLD)
    if not pdr_result["found"]:
        return {"answer": "🚨 No reliable information found for this question.",
                "source": "guard", "intent": intent, "cache_hit": False,
                "latency_ms": round((time.perf_counter() - t0) * 1000, 1)}

    answer = simulate_llm(question, pdr_result["full_context"])
    cache.store(prompt=question, response=answer,
                metadata={"parent_id": pdr_result["parent_id"]})

    return {"answer": answer, "source": pdr_result["parent_id"],
            "confidence": pdr_result["confidence"], "intent": intent,
            "cache_hit": False,
            "latency_ms": round((time.perf_counter() - t0) * 1000, 1)}

# Run the full pipeline
for q in [
    "Does the Aether X1 have good graphics card?",
    "What is the Aether X1 GPU?"  ,              # ← should HIT cache (rephrasing)
    "Does our solar panel charge the battery?"   # ← should GUARD (out of scope)
]:
    r = production_rag(q)
    status = '💚 CACHE' if r['cache_hit'] else ('🔴 GUARD' if r['source']=='guard' else '🟡 PDR')
    print(f"{status} | {r['latency_ms']:>7.1f}ms | {q[:55]}")
    print(f"  Answer: {r['answer'][:90]}...\n")

🟡 PDR |   644.8ms | Does the Aether X1 have good graphics card?
  Answer: Based on our documentation: The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU ...

💚 CACHE |    30.6ms | What is the Aether X1 GPU?
  Answer: Based on our documentation: The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU ...

🔴 GUARD |    30.2ms | Does our solar panel charge the battery?
  Answer: 🚨 No reliable information found for this question....



In [68]:
# Cell 12: Checkpoint
score = 0

r_hit  = production_rag("Aether X1 price?")  # should cache hit
r_guard = production_rag("what is the weather today?")  # should guard
r_pdr  = production_rag("What is the Sony battery life?")  # should PDR

if r_hit['cache_hit']:
    print("✅ Test 1 PASS: Semantic cache HIT for rephrased question")
    score += 1
else:
    print("❌ Test 1 FAIL: Cache should have hit for price question")

if r_guard['source'] == 'guard':
    print("✅ Test 2 PASS: Out-of-scope question correctly blocked by guard")
    score += 1
else:
    print("❌ Test 2 FAIL: Guard did not block out-of-scope question")

if r_pdr['source'] not in ['cache', 'guard']:
    print("✅ Test 3 PASS: PDR retrieved Sony XM5 document")
    score += 1
else:
    print("❌ Test 3 FAIL: Expected PDR retrieval, got", r_pdr['source'])

print(f"\n🏆 Score: {score}/3")

✅ Test 1 PASS: Semantic cache HIT for rephrased question
✅ Test 2 PASS: Out-of-scope question correctly blocked by guard
✅ Test 3 PASS: PDR retrieved Sony XM5 document

🏆 Score: 3/3


In [69]:
r_pdr

{'answer': 'Based on our documentation: The Sony WH-1000XM5 headphones feature industry-leading active noise cancellation with dual-chip processing. Battery life is 30 hours with ANC on, 40 hours without. Supports LDAC, AAC, and SBC codecs....',
 'source': 'doc_sony_xm5',
 'confidence': 0.5362,
 'intent': 'warranty',
 'cache_hit': False,
 'latency_ms': 552.7}

---
## 📝 Assignment: Multi-Index RAG with Routing

**Task 1**: Extend `production_rag()` to route to **different PDR indices** based on intent.
  - `product_specs` & `pricing` → search `pdr_index` (product catalog)
  - `shipping` & `returns` → search a second `logistics_pdr_index`
  - `warranty` → search a third `warranty_pdr_index`

**Task 2**: Add a **cache hit rate tracker**:
  - Run 20 questions (mix of unique + repeats)
  - Print: total_requests, cache_hits, cache_hit_rate, avg_latency_ms

**Task 3 (Bonus)**: Tune `GUARD_THRESHOLD` from 0.2 → 0.5
  - For each threshold, run 10 questions and count: hits, misses, guard_blocks
  - Plot the tradeoff (use print table if matplotlib unavailable)


In [70]:
# ── ASSIGNMENT ────────────────────────────────────────────────────────────────

# Task 1: Multi-index routing
class MultiIndexRAG:
    """
    TODO: RAG pipeline with separate indices per domain.

    Each domain has its own PDR index:
      - 'product'   : product specs, pricing
      - 'logistics' : shipping, returns
      - 'support'   : warranty, technical_support

    Methods:
      ingest(parent_id, text, domain)  → index into correct PDR index
      query(question)                  → route → cache → PDR → guard → answer
    """
    def __init__(self, redis_url: str):
        # TODO: create 3 SearchIndex instances (product, logistics, support)
        # TODO: create one shared SemanticCache
        pass

    def ingest(self, parent_id: str, text: str, domain: str):
        # TODO: decompose text into propositions, embed, load into domain index
        pass

    def query(self, question: str) -> dict:
        # TODO: detect_intent → map to domain → cache check → PDR → guard
        pass


# Task 2: Cache hit rate tracker
def cache_hit_rate_test(rag_system, questions: list) -> dict:
    """
    TODO:
    - Run all questions through rag_system.query()
    - Track: total, hits, misses, guard_blocks
    - Return summary dict with cache_hit_rate and avg_latency_ms
    """
    pass


# Task 3 (Bonus): Threshold sweep
def threshold_sweep(questions: list, thresholds: list) -> None:
    """
    TODO: For each threshold in [0.2, 0.3, 0.4, 0.5]:
      - Re-configure the guard threshold
      - Run all questions
      - Count: PDR_hits, guard_blocks
      - Print a table:
        Threshold | PDR Hits | Guard Blocks
        0.20      |  3       |  7
        0.30      |  6       |  4
        ...
    Explain: which threshold gives the best balance?
    """
    pass


print("📝 Assignment ready — implement MultiIndexRAG and cache_hit_rate_test")

📝 Assignment ready — implement MultiIndexRAG and cache_hit_rate_test
